# Part 11: Web Development for Data Scientists (Flask)
Flask is a lightweight web framework for Python. It is the industry standard for Data Scientists who need to quickly deploy Machine Learning models as Web APIs or interactive dashboards.



## 1. HTML, CSS, & Jinja Templates
* **HTML:** Structures the web page (forms, buttons, text).
* **CSS:** Styles the web page (colors, layouts).
* **Jinja2:** A templating engine that allows you to inject Python variables directly into your HTML files using `{{ variable }}` or run logic using `{% if %}`.

In [4]:
import os

# Create a folder to hold our HTML templates
os.makedirs('templates', exist_ok=True)

# Create a folder for static files (CSS, Images)
os.makedirs('static', exist_ok=True)
print("✅ Folders 'templates/' and 'static/' created.")

✅ Folders 'templates/' and 'static/' created.


In [5]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>ML Prediction API</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9; }
        .container { background: white; padding: 20px; border-radius: 8px; box-shadow: 0px 0px 10px rgba(0,0,0,0.1); }
        .flash { padding: 10px; margin-bottom: 20px; border-radius: 5px; background-color: #d4edda; color: #155724; }
    </style>
</head>
<body>
    <div class="container">
        <h1>House Price Estimator 🏠</h1>
        
        {% with messages = get_flashed_messages() %}
            {% if messages %}
                {% for message in messages %}
                    <div class="flash">{{ message }}</div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        <form action="/predict" method="POST">
            <label for="area">Area (Sq Ft):</label><br>
            <input type="number" id="area" name="area" required><br><br>
            
            <label for="bedrooms">Bedrooms:</label><br>
            <input type="number" id="bedrooms" name="bedrooms" required><br><br>
            
            <button type="submit">Predict Price</button>
        </form>
        
        {% if prediction %}
            <h2>Estimated Price: ₹{{ prediction }} Lakhs</h2>
        {% endif %}
    </div>
</body>
</html>

Overwriting templates/index.html


## 2. The Flask Framework & Creating APIs
An **API (Application Programming Interface)** allows other programs to send data to your server and get a prediction back, usually in JSON format.

In this app, we will build:
1. **A Web Route (`/`)**: Serves the HTML form.
2. **A Form Handler (`/predict`)**: Processes the HTML form submission using `POST`.
3. **An API Endpoint (`/api/v1/predict`)**: Accepts **Query Parameters** (e.g., `?area=1500&beds=3`) and returns raw JSON data.

In [6]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify, flash, redirect, url_for

# Initialize the Flask application
app = Flask(__name__)
app.secret_key = "super_secret_key_for_flash_messages" # Required for flash messages

# 1. Standard Web Route (Serves HTML)
@app.route('/', methods=['GET'])
def home():
    return render_template('index.html')

# 2. Form Handling Route
@app.route('/predict', methods=['POST'])
def predict_ui():
    try:
        # Extract data from the HTML form
        area = float(request.form['area'])
        bedrooms = int(request.form['bedrooms'])
        
        # Simulated Model Prediction logic
        # (In reality, you would use joblib to load the model you saved earlier: model.predict([[area, bedrooms]]))
        estimated_price = round((area * 0.05) + (bedrooms * 2.5), 2)
        
        # Flash a success message
        flash("Prediction generated successfully!")
        
        # Render the template again, passing the prediction variable via Jinja
        return render_template('index.html', prediction=estimated_price)
    
    except Exception as e:
        flash(f"Error: {str(e)}")
        return redirect(url_for('home'))

# 3. Creating an API with Query Parameters
# Try accessing: http://127.0.0.1:5000/api/v1/predict?area=2000&beds=4
@app.route('/api/v1/predict', methods=['GET'])
def predict_api():
    # Extract Query Parameters from the URL
    area = request.args.get('area', type=float)
    beds = request.args.get('beds', type=int)
    
    if area is None or beds is None:
        return jsonify({'error': 'Please provide both area and beds parameters.'}), 400
        
    # Simulated Prediction
    estimated_price = round((area * 0.05) + (beds * 2.5), 2)
    
    # APIs return JSON data, not HTML
    return jsonify({
        'status': 'success',
        'input': {'area_sqft': area, 'bedrooms': beds},
        'estimated_price_lakhs': estimated_price
    })

if __name__ == '__main__':
    # use_reloader=False is important when running inside certain IDE environments
    app.run(debug=True, port=5000, use_reloader=False)

Overwriting app.py


## 3. Running the Application
Because a web server runs continuously, running it inside a Jupyter Code cell will "block" the cell until you shut the server down. 

To test your new web app:
1. Open your **VS Code Terminal** (Git Bash).
2. Ensure you are in your project directory (`D:\DAS`).
3. Run the following command:
   ```bash
   python app.py